# Evaluating a reinsurance submission with prospicio

A broker sends a reinsurance submission. A cedant (the insurer that buys reinsurance) asks for quotes on its 2026 **per-risk excess of loss** program for commercial property. This notebook follows the work of a reinsurer's pricing actuary on that pack. It starts from the raw files and ends with a decision for each layer: quote, counter, decline or refer.

The notebook is for an actuary who prices primary insurance and has not priced a treaty. It defines each term where it first occurs. The guide [What a reinsurance price is](../docs/guides/reinsurance-price.md) explains the pricing concepts in more detail (layers, rate on line, capital, technical premium). `reinsurance_pricing.ipynb` checks the layer calculations against another package.

**The case.** The cedant is a synthetic commercial property insurer. It asks for three layers of cover on each risk (a *risk* is one insured location). The treaty is on a **losses occurring** basis for 2026: it covers losses that occur during 2026. The layers are 5m xs 5m, 10m xs 10m and 20m xs 20m. "5m xs 5m" means 5 million excess of 5 million: the layer pays the part of a loss between 5m and 10m. Each layer has paid reinstatements and a **rate on line** (premium divided by limit) that the broker quotes. The submission has the files that a broker sends, in `examples/data/submission/`:

| File | What it holds |
|---|---|
| `subject_premium.csv`, `rate_changes.csv` | Subject premium (the premium the treaty applies to) by year, and the cedant's rate changes |
| `cost_index.csv` | A published building cost index, the source for claims inflation |
| `large_losses.csv` | Every claim over 500,000 incurred, valued at each year end: paid and incurred amounts, policy limit, dates |
| `triangle.csv` | The whole book's paid and incurred loss triangle |
| `risk_profile.csv` | The 2026 portfolio by band of sum insured: number of risks, sum insured, premium |
| `treaty_terms.csv`, `treaty_program.csv` | The proposed layers and quoted rates, and the treaty period, basis and brokerage |

The data is synthetic. `examples/data/make_submission.py` made it from known parameters (a Pareto tail, a claim frequency, trends). Thus at the end we can compare the estimates with the truth. The answer key, `examples/data/answer_key.json`, is outside the submission's folder, and the notebook reads it only in section 14.

**What a real submission adds.** A real pack has these items, and each needs its own step:

* a split of catastrophe claims from per-risk claims (a per-risk treaty excludes them or models them separately).
* the treatment of loss adjustment expenses (included in the loss, pro rata, or in addition).
* the definition of a risk, and of a loss when one event damages several risks.
* the currency and any index clause on the attachment (`LargeLossExperience(fx=)` and `IndexClause`). This case has neither.
* a tail factor after the oldest age of the triangle.
* premium that increases with insured values. Here the on-level premium increases only with the number of risks, while sums insured and claims increase with the cost index (section 2).

| Section | Step |
|---|---|
| 1 | Load the files and reconcile them with each other |
| 2 | On-level premium and trend |
| 3 | Develop the book's triangle to ultimate |
| 4 | Prepare the large losses: development, trend, completeness |
| 5 | Fit the severity and the frequency of large losses |
| 6 | Experience rating: the cost of each layer from the cedant's own claims |
| 7 | Exposure rating: the cost of each layer from the cedant's portfolio |
| 8 | Blend the two by credibility, layer by layer |
| 9 | Build the loss model and apply the treaty terms |
| 10 | Risk metrics for each layer |
| 11 | Technical premium, and the broker's quotes against it |
| 12 | Sensitivity and parameter uncertainty |
| 13 | The decision |
| 14 | How close the estimates were to the truth |

To run the notebook, clone the repository and start Jupyter with prospicio built from source, which needs Rust. From the repository root, run `uv run --project python --group examples jupyter lab` (see `examples/README.md`), then open this notebook. Set `PROSPICIO_EXAMPLE_SIMS` to run fewer simulations. The docs site shows the results. The repository keeps only the code.

In [ ]:
import csv
import json
import math
import os
from collections import defaultdict
from pathlib import Path

try:
    from prospicio.credibility import LimitedFluctuation
    from prospicio.distributions import Grid, Pareto, PolicyLimits, claim_count
    from prospicio.models import Glm, Terms
    from prospicio.pricing import (CollectiveModel, DevelopmentPattern, LargeLossExperience, Mbbefd, Period, Quote,
                                   RateHistory, RiskProfile, TrendFit, credibility_weighted, fit_references, layer_cv,
                                   price, trend_factor, year_fraction)
    from prospicio.reinsurance import Layer, Tower, occurrence_layer_statistics
    from prospicio.reserving import CapeCod, ChainLadder, Triangle
    from prospicio.risk import Distortion, calibrate
except ImportError as error:
    raise ImportError("this notebook needs a prospicio newer than 0.0.2 (the latest release on PyPI): build it from "
                      "source, `uv run --project python --group examples jupyter lab` from the repository root") from error

N_SIMS = int(os.environ.get("PROSPICIO_EXAMPLE_SIMS", "20000"))

# The submission's folder, from wherever the notebook runs inside the repository.
DATA = next((p / sub for p in [Path.cwd(), *Path.cwd().parents]
             for sub in ("examples/data/submission", "data/submission") if (p / sub).is_dir()), None)
if DATA is None:
    raise FileNotFoundError(
        f"examples/data/submission not found in {Path.cwd()} or above it: open this notebook from inside a clone "
        "of the repository (examples/data/make_submission.py writes the files)")


def read_csv(name):
    with open(DATA / name) as f:
        return list(csv.DictReader(f))


def when(text):
    """A date 'YYYY-MM-DD' as a year fraction (the library's half-month grid)."""
    y, m, d = map(int, text.split("-"))
    return year_fraction(y, m, d)


def money(x):
    return f"{x / 1e6:,.2f}m"


print(f"submission files in {DATA.parent.name}/{DATA.name}: {', '.join(sorted(p.name for p in DATA.glob('*.csv')))}")

## 1. The submission

First, read the files and print the treaty terms. Money is in US dollars. The cedant lists each claim whose incurred loss is over 500,000 at the valuation date, 31 December 2025. The incurred loss is paid plus the case reserve that the cedant's claims department sets. Each claim has a row for each year-end valuation since its report. This history lets us measure how claims develop.

In [ ]:
program = {r["item"]: r["value"] for r in read_csv("treaty_program.csv")}
for item, value in program.items():
    print(f"{item:36s} {value}")

terms = [{"name": r["layer"], "limit": float(r["limit"]), "attachment": float(r["attachment"]),
          "reinstatements": int(r["reinstatements"]), "reinstatement_rate": float(r["reinstatement_rate"]),
          "quoted_rol": float(r["quoted_rate_on_line"])} for r in read_csv("treaty_terms.csv")]
print()
for t in terms:
    print(f"{t['name']:>11s}: {t['reinstatements']} paid reinstatement(s) at {t['reinstatement_rate']:.0%}, "
          f"broker's quote {t['quoted_rol']:.1%} rate on line, deposit {money(t['quoted_rol'] * t['limit'])}")

LISTING_THRESHOLD = float(program["listing_threshold"])
BROKERAGE = float(program["brokerage"])
GNPI = float(program["estimated_subject_premium_income"])

premium_rows = read_csv("subject_premium.csv")
YEARS = [int(r["calendar_year"]) for r in premium_rows]
SUBJECT_PREMIUM = {int(r["calendar_year"]): float(r["subject_premium"]) for r in premium_rows}

# The listing: one row per claim and valuation date.
claims = {}
for r in read_csv("large_losses.csv"):
    c = claims.setdefault(r["claim_id"], {"year": int(r["accident_date"][:4]), "accident": when(r["accident_date"]),
                                          "report": int(r["report_date"][:4]), "limit": float(r["policy_limit"]),
                                          "report_date": r["report_date"], "accident_date": r["accident_date"],
                                          "values": {}})
    c["values"][int(r["valuation_date"][:4])] = (float(r["paid"]), float(r["incurred"]))

triangle_rows = read_csv("triangle.csv")
profile_rows = read_csv("risk_profile.csv")

**Reconcile first.** People make a submission from several systems, and its files rarely agree at first. Before you model, do these checks:

1. The claim listing agrees with itself.
2. The listing fits inside the triangle. The triangle includes all claims, so its latest diagonal must be at least the listed claims.
3. The triangle's paid is never more than its incurred.
4. The premium in the risk profile is near the premium that the cedant expects for 2026.

A failed check is a question for the broker. Do not correct it without telling the broker. The table shows the share of each year's incurred loss that is in the listing. This is the part of the loss that the large-loss model carries.

In [ ]:
checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'ok  ' if ok else 'FAIL'} {name}" + (f": {detail}" if detail else ""))


last = {cid: max(c["values"]) for cid, c in claims.items()}
check("every claim has a valuation at each year end from its report year to 2025",
      all(sorted(c["values"]) == list(range(c["report"], 2026)) for c in claims.values()))
check("incurred is at least paid, and at most the policy limit",
      all(p <= i + 1e-6 and i <= c["limit"] + 1 for c in claims.values() for p, i in c["values"].values()))
check("the last valuation of every claim is over the listing threshold",
      all(c["values"][last[cid]][1] >= LISTING_THRESHOLD for cid, c in claims.items()),
      f"smallest {money(min(c['values'][last[cid]][1] for cid, c in claims.items()))}")
check("no claim is reported before it happens", all(c["report_date"] >= c["accident_date"] for c in claims.values()))

# The triangle holds the listed claims too: its latest diagonal must be at least their incurred.
latest_triangle = {}
for r in triangle_rows:
    y, age = int(r["accident_year"]), int(r["development_months"])
    if age == 12 * (2026 - y):
        latest_triangle[y] = float(r["incurred"])
large_incurred = defaultdict(float)
for cid, c in claims.items():
    large_incurred[c["year"]] += c["values"][last[cid]][1]
check("the triangle's latest diagonal covers the listed claims",
      all(latest_triangle[y] >= large_incurred[y] for y in YEARS))
cells = {(int(r["accident_year"]), int(r["development_months"])): (float(r["paid"]), float(r["incurred"]))
         for r in triangle_rows}
check("the triangle's paid is at most its incurred in every cell", all(p <= i for p, i in cells.values()))
# Incurred that falls with age is possible (case reserves revised down) but worth a question to the broker.
falls = [(y, age) for (y, age), (_, i) in cells.items() if (y, age - 12) in cells and i < cells[y, age - 12][1]]
print("     incurred falls from one age to the next in " + (", ".join(
    f"{y} at {age} months ({money(cells[y, age][1] - cells[y, age - 12][1])})" for y, age in falls) or "no cell"))

profile_premium = sum(float(r["premium"]) for r in profile_rows)
check("the risk profile's premium is near the estimated 2026 subject premium",
      abs(profile_premium / GNPI - 1) < 0.05, f"profile {money(profile_premium)}, estimate {money(GNPI)}")
check("the bands of the profile are contiguous",
      all(profile_rows[i]["upper_sum_insured"] == profile_rows[i + 1]["lower_sum_insured"] for i in range(len(profile_rows) - 1)))

print(f"\n{len(claims)} claims in the listing, {len(read_csv('large_losses.csv'))} valuation rows; "
      f"{sum(ok for _, ok in checks)} of {len(checks)} checks pass")
print(f"\n{'year':>6s} {'subject premium':>16s} {'incurred (latest)':>18s} {'listed claims':>14s} {'share':>7s}")
for y in YEARS:
    print(f"{y:>6d} {money(SUBJECT_PREMIUM[y]):>16s} {money(latest_triangle[y]):>18s} "
          f"{money(large_incurred[y]):>14s} {large_incurred[y] / latest_triangle[y]:7.1%}")

## 2. On-level premium and trend

**On-level premium.** Premium increased for two reasons: more exposure and higher rates. Rate changes take years to reach earned premium, because policies are written during the year and earn over the next twelve months. To compare years, we restate the premium of each year at the *current* rate level. This is the **on-level premium**, by the parallelogram method. `RateHistory` holds the cedant's rate changes, and `rate_level` gives the factor for a period. The treaty covers losses occurring, so the periods are calendar years of earned premium. After on-levelling, the remaining change is exposure, which we use to scale claim counts.

**Trend.** Claims cost more as prices increase. We *trend* each loss from its date of occurrence to the average date of the treaty year, so all years are at 2026 prices. The source is the building cost index, fitted with an exponential `TrendFit`. This is a judgement: the cedant's own claims inflation can be different from the index, so section 12 tests it.

Premium is not trended. In this case the on-level premium increases only with the number of risks, while sums insured and claims increase with the index. Thus the loss ratio increases with the full cost trend. On a real book, insured values usually increase too, and with them premium charged as a rate on value. The trend of the loss ratio is then the cost trend less the premium trend.

In [ ]:
rate_rows = read_csv("rate_changes.csv")
history = RateHistory([when(r["effective_date"]) for r in rate_rows],
                      [float(r["average_rate_change"]) for r in rate_rows])
onlevel_factor, onlevel = {}, {}
for y in YEARS:
    level = history.rate_level(Period.calendar(y, y + 1), 1.0)
    onlevel_factor[y] = level.on_level_factor
    onlevel[y] = SUBJECT_PREMIUM[y] * onlevel_factor[y]
print(f"current rate level {history.current_level:.4f} (1 before the first change)")
print(f"{'year':>6s} {'premium':>10s} {'on-level factor':>16s} {'on-level premium':>17s}")
for y in YEARS:
    print(f"{y:>6d} {money(SUBJECT_PREMIUM[y]):>10s} {onlevel_factor[y]:>16.4f} {money(onlevel[y]):>17s}")

# On-level premium moves with exposure only, so its trend is the exposure growth.
exposure_trend = TrendFit.exponential([y + 0.5 for y in YEARS], [onlevel[y] for y in YEARS])
TREATY_MIDDLE = Period.calendar(2026, 2027).average_earned_date(1.0)  # average accident date of the treaty year
ONLEVEL_2026 = exposure_trend.fitted(TREATY_MIDDLE)
print(f"\nexposure growth {exposure_trend.annual_change:.2%} a year (R-squared {exposure_trend.r_squared:.4f}); "
      f"on-level premium for 2026: {money(ONLEVEL_2026)}")
print(f"the risk profile's premium is {money(profile_premium)} ({profile_premium / ONLEVEL_2026 - 1:+.1%}), "
      f"the cedant's estimate of 2026 subject premium {money(GNPI)} (at the rates in force during 2026)")

# Severity trend from the published building cost index.
index_rows = read_csv("cost_index.csv")
cost_trend = TrendFit.exponential([when(r["quarter_start"]) + 0.125 for r in index_rows],
                                  [float(r["index"]) for r in index_rows])
SEVERITY_TREND = cost_trend.annual_change
print(f"\nbuilding cost index trend {SEVERITY_TREND:.2%} a year (R-squared {cost_trend.r_squared:.4f}, "
      f"{cost_trend.n} quarters)")


def trend_to_treaty(year_or_date):
    """Factor that moves a loss at an accident date (or the middle of an accident year) to the treaty year's prices."""
    t = year_or_date + 0.5 if isinstance(year_or_date, int) else year_or_date
    return trend_factor(SEVERITY_TREND, TREATY_MIDDLE - t)


print(f"a loss from the middle of 2016 is trended by {trend_to_treaty(2016):.3f}, one from 2025 by {trend_to_treaty(2025):.3f}")

## 3. Develop the book triangle

The triangle holds the cumulative paid and incurred loss of the whole book by accident year and age in months. Recent years are not complete, so we develop each year to **ultimate** (its final cost). `ChainLadder` multiplies the latest value of each year by the factor to ultimate from the older years. **Cape Cod** is a Bornhuetter-Ferguson method that estimates its prior loss ratio from the triangle, with on-level premium as the exposure. Its `trend` argument moves the loss ratio of each year to a common price level.

This step gives the **ground-up expected loss ratio** at 2026 prices and rates: the expected cost of all claims, of all sizes, as a share of premium. The exposure rating in section 7 spreads that loss across the sums insured of the portfolio. It is not the cost of a layer, which is a share of it.

Paid and incurred chain ladders give a cross-check. But they use the same triangle as Cape Cod, so their agreement shows little about how firm the selection is. The scatter of the trended loss ratios of the ten years shows more. Section 12 moves the loss ratio by about two standard errors of its ten-year average.

In [ ]:
triangle = Triangle.from_long(
    [int(r["accident_year"]) for r in triangle_rows],
    [int(r["development_months"]) for r in triangle_rows],
    {"incurred": [float(r["incurred"]) for r in triangle_rows],
     "paid": [float(r["paid"]) for r in triangle_rows],
     "onlevel": [onlevel[int(r["accident_year"])] for r in triangle_rows]},
)
chain = ChainLadder().fit(triangle, "incurred")
paid_chain = ChainLadder().fit(triangle, "paid")
print("age-to-age factors (incurred):", ", ".join(f"{f:.3f}" for f in chain.ldf[:5]), "...")
print("                       (paid):", ", ".join(f"{f:.3f}" for f in paid_chain.ldf[:5]), "...")

cape_cod = CapeCod(trend=SEVERITY_TREND).fit(triangle, "incurred", "onlevel")


def loss_ratio(ultimates, trend=SEVERITY_TREND):
    """Ultimates trended to the middle of the treaty year over on-level premium, pooled across the years."""
    return (sum(u * trend_factor(trend, TREATY_MIDDLE - (y + 0.5)) for u, y in zip(ultimates, YEARS))
            / sum(onlevel[y] for y in YEARS))


latest = dict(zip(YEARS, chain.latest))
ult_chain = {y: l + r for y, l, r in zip(YEARS, chain.latest, chain.reserve)}
ult_paid = {y: l + r for y, l, r in zip(YEARS, paid_chain.latest, paid_chain.reserve)}
ult_cc = dict(zip(YEARS, cape_cod.ultimate))
yearly = {y: ult_cc[y] * trend_to_treaty(y) / onlevel[y] for y in YEARS}
print(f"\n{'year':>6s} {'incurred':>10s} {'chain ladder':>13s} {'paid CL':>9s} {'Cape Cod':>9s} "
      f"{'loss ratio, CC, trended':>24s}")
for y in YEARS:
    print(f"{y:>6d} {money(latest[y]):>10s} {money(ult_chain[y]):>13s} {money(ult_paid[y]):>9s} "
          f"{money(ult_cc[y]):>9s} {yearly[y]:>24.1%}")

# The selected loss ratio: the Cape Cod ultimates pooled, which is Cape Cod's own estimate across the years
# moved to the treaty year's prices (its `trended_apriori` is at the valuation date's prices, a year earlier).
ELR = loss_ratio(cape_cod.ultimate)
spread = math.sqrt(sum((v - ELR) ** 2 for v in yearly.values()) / (len(YEARS) - 1))
print(f"\nselected ground-up loss ratio at 2026 prices and rates: {ELR:.1%} "
      f"(chain ladder {loss_ratio(ult_chain.values()):.1%}, paid chain ladder {loss_ratio(ult_paid.values()):.1%})")
print(f"the years run from {min(yearly.values()):.1%} to {max(yearly.values()):.1%}: a standard deviation of "
      f"{spread:.1%}, so about {spread / math.sqrt(len(YEARS)):.1%} on the ten-year average")

## 4. Prepare the large losses

An excess of loss layer pays only for large claims. Its price thus comes from the cedant's large-loss listing, and that listing needs more care than any other file.

* **Development.** A claim's incurred amount usually changes after its first report. This movement is IBNER (incurred but not enough reported). The helper `incurred_development` measures it from the listing. For each development year, it divides the sum of the incurred amounts one year later by the sum now, for the claims valued at both ages. `LargeLossExperience.prepare` then develops and trends each claim with its own factor to ultimate, capped so that the claim does not go above its policy limit.
  The listing holds the claims whose *latest* incurred is over 500,000. A claim that was large at an early valuation and later went below the listing threshold is not in it. Factors measured on all listed claims thus have an upward bias. `DevelopmentPattern.from_history` uses all claims valued at both ages. The helper uses only the claims whose incurred at the *earlier* valuation was already over the analysis threshold below. That threshold is far above the listing threshold, so few of these claims can have left the listing. The cell prints both sets of factors. In this listing they differ by less than one point.
* **Completeness.** The listing includes claims over 500,000 at the latest valuation. But a claim just over that amount now can end well above it. A claim just under it can be missing although it will end over it. Only claims over an **analysis threshold** above the listing threshold (750,000 here) are complete. The margin is not perfect. A claim that occurred in the last weeks of 2025 can have a reserve at two thirds of its ultimate. Thus a claim that will end at 760,000 can still be under 500,000 and missing. Section 12 moves the analysis threshold to 1m to show the effect.
* **Trend and the reporting threshold.** The threshold is in the money of each accident year. A claim of 750,000 in the first accident year is worth much more at 2026 prices. Thus, after the trend, the cut for the older years is higher than the cut for recent years (the table below shows the cut of each year). Each claim carries its own cut in treaty-year prices. The Pareto fit in section 5 uses these cuts as left truncation points. The policy limit is trended with the claim, on the assumption that sums insured follow the same index.
* **Past structures.** We assume that the cedant's structure did not change during the experience period, because the submission gives only the current one. When a submission gives past structures, `PreparedLosses.as_if(historical, years, proposed)` gives the recoveries of each year under the program in force then, next to the proposed tower. Use it to reconcile with the recoveries that the cedant reports.

In [ ]:
ANALYSIS_THRESHOLD = 750_000.0  # complete from here, at the prices of each accident year


def incurred_development(claims, min_incurred, max_year=4):
    """HELPER (`DevelopmentPattern.from_history` has no threshold on the earlier valuation). Age-to-age
    incurred factors from the listing itself: for each development year d (valuation year minus accident year),
    the sum of the incurred at d + 1 over the sum at d, over the claims valued at both whose incurred at d is at
    least `min_incurred` (the listing is selected on the latest incurred; a claim large enough at d is in it
    whatever its later development).
    Returns the factors and the cumulative factors to ultimate, by development year, with no development after
    `max_year`."""
    num, den = defaultdict(float), defaultdict(float)
    for c in claims.values():
        for v, (_, inc) in c["values"].items():
            if v + 1 in c["values"] and inc >= min_incurred:
                num[v - c["year"]] += c["values"][v + 1][1]
                den[v - c["year"]] += inc
    link = {d: num[d] / den[d] for d in sorted(num) if d <= max_year}
    to_ultimate, run = {}, 1.0
    for d in range(max(YEARS) - min(YEARS) + 1, -1, -1):
        to_ultimate[d] = run
        run *= link.get(d - 1, 1.0)
    return link, to_ultimate


link, to_ultimate = incurred_development(claims, ANALYSIS_THRESHOLD)
print(f"age-to-age incurred factors of the claims over {money(ANALYSIS_THRESHOLD)} at the earlier valuation:",
      ", ".join(f"{d}: {f:.3f}" for d, f in link.items()))
# The library's factors: every claim valued at both ages, by development year (the listing's selection stays in).
valuations = [(cid, v - c["year"], inc) for cid, c in claims.items() for v, (_, inc) in c["values"].items()]
every_claim = DevelopmentPattern.from_history([cid for cid, _, _ in valuations],
                                              [float(d) for _, d, _ in valuations], [inc for _, _, inc in valuations],
                                              ages=[float(d) for d in range(len(YEARS))])
print("  measured on every listed claim, they would be:",
      ", ".join(f"{d}: {f:.3f}" for d, f in enumerate(every_claim.age_to_age[:len(link)])))

# Every claim at its latest incurred (valued 31 December 2025), developed to ultimate and trended to the treaty
# year. A claim cannot develop past its policy limit, so its factor to ultimate carries the cap.
ids = list(claims)
reported = [claims[cid]["values"][last[cid]][1] for cid in ids]
to_limit = [min(r * to_ultimate[last[cid] - claims[cid]["year"]], claims[cid]["limit"]) / r
            for cid, r in zip(ids, reported)]
record = LargeLossExperience(ids, [claims[cid]["accident"] for cid in ids], reported,
                             valuation=year_fraction(2025, 12, 31))
prepared = record.prepare(rate=SEVERITY_TREND, to=TREATY_MIDDLE, development=to_limit)
table = prepared.to_dict()  # one row per claim, in the record's order: ultimate, trend factor, trended loss
large = [{"id": cid, "year": claims[cid]["year"], "reported": r, "ultimate": u, "limit": claims[cid]["limit"],
          "trend": f, "loss": loss, "at_limit": u >= 0.999 * claims[cid]["limit"]}
         for cid, r, u, f, loss in zip(table["id"], reported, table["ultimate"], table["factor"], table["trended"])]

# Complete claims: developed ultimate over the analysis threshold at the prices of the accident date.
eligible = [x for x in large if x["ultimate"] >= ANALYSIS_THRESHOLD]
for x in eligible:
    # x["loss"] is at the treaty year's prices, and the policy limit is trended with it.
    x["threshold"] = ANALYSIS_THRESHOLD * x["trend"]  # the same cut, in treaty-year prices: it differs by year
    x["censored"] = x["at_limit"]
print(f"\n{len(large)} listed claims, {len(eligible)} over {money(ANALYSIS_THRESHOLD)} once developed; "
      f"{sum(x['censored'] for x in eligible)} of them at their policy limit")
print(f"{'year':>6s} {'listed':>7s} {'complete':>9s} {'trend':>7s} {'cut in 2026 prices':>19s} {'largest, 2026 prices':>21s}")
for y in YEARS:
    ys = [x for x in eligible if x["year"] == y]
    print(f"{y:>6d} {sum(x['year'] == y for x in large):>7d} {len(ys):>9d} {trend_to_treaty(y):>7.3f} "
          f"{money(ANALYSIS_THRESHOLD * trend_to_treaty(y)):>19s} {money(max((x['loss'] for x in ys), default=0)):>21s}")

**Claims not yet reported.** Some large claims occurred but are not yet in the listing. A triangle of claim *counts* by accident year and year of report, developed with the same chain ladder, gives a factor on the count of each year. The factor is large only for the most recent year. `burning_cost` in section 6 reads the same factors as a `DevelopmentPattern` by age, at the average age of each year.

In [ ]:
# Late-reported claims: a triangle of claim counts by report year, developed with the same chain ladder.
reported_in = defaultdict(int)
for c in claims.values():
    reported_in[(c["year"], c["report"])] += 1
rows = [(y, 12 * (v - y + 1), float(sum(reported_in[(y, w)] for w in range(y, v + 1))))
        for y in YEARS for v in range(y, 2026)]
counts = Triangle.from_long([r[0] for r in rows], [r[1] for r in rows], {"claims": [r[2] for r in rows]})
count_fit = ChainLadder().fit(counts, "claims")
count_factor = {y: (count_fit.latest[i] + count_fit.reserve[i]) / count_fit.latest[i] for i, y in enumerate(YEARS)}
# The same factors by age, for the burning cost: an accident year is on average half a year old at its year end.
count_pattern = DevelopmentPattern([0.5 + k for k in range(len(count_fit.cdf))], list(count_fit.cdf))
print("claims still to be reported, as a factor on the count:",
      ", ".join(f"{y}: {count_factor[y]:.3f}" for y in YEARS[-3:]))

## 5. Severity and frequency of the large losses

**Severity.** A **Pareto** distribution often describes large losses above a threshold well. The probability of a loss above `x` is `(T / x) ** alpha` for `x` over the threshold `T`. A lower `alpha` gives a heavier tail. `Pareto.fit` finds the maximum likelihood alpha and includes the two data features of the listing:

* **truncation**: a claim is in the data only if it was over the cut of its year (the `reporting_thresholds`).
* **censoring**: a claim at its policy limit is known only to be at least that large (the `censored` flags).

The three values below show the effect of each feature. If you ignore the thresholds, the fit assumes that the older years show all claims over `T`. But those years show only claims over their higher cut, so the fitted tail is much heavier. `Pareto.fit_with_standard_error` also returns the standard error of the alpha, from the observed information. Without truncation, it is alpha over the square root of the number of uncensored claims. The thresholds increase it a little.

In [ ]:
T = ANALYSIS_THRESHOLD
losses = [x["loss"] for x in eligible]
thresholds = [x["threshold"] for x in eligible]
censored = [x["censored"] for x in eligible]

severity_fit, ALPHA_SE = Pareto.fit_with_standard_error(losses, T, reporting_thresholds=thresholds, censored=censored)
ALPHA = severity_fit.alpha
n_uncensored = sum(not c for c in censored)
print(f"Pareto alpha with truncation and censoring: {ALPHA:.3f} (standard error {ALPHA_SE:.2f}; "
      f"alpha / sqrt(uncensored claims) is {ALPHA / math.sqrt(n_uncensored):.2f})")
print(f"  ignoring the reporting thresholds:        {Pareto.fit(losses, T, censored=censored).alpha:.3f}")
print(f"  ignoring the policy limits as well:       {Pareto.fit(losses, T).alpha:.3f}")

**Frequency.** The number of claims over `T` per year must be on a common basis. The calculation has three steps:

1. Gross up the counts for claims not yet reported.
2. Move each count from the cut of its year to `T`, with the fitted alpha.
3. Divide by the exposure of the year relative to 2026.

A Poisson GLM with that adjustment as an offset does the calculation. It also tests for a trend that remains in the counts after the exposure and the cut. Its Pearson dispersion measures over-dispersion. A value over 1 shows that the counts change more from year to year than a Poisson count, and we then use a negative binomial. With ten years of counts that estimate is weak, so section 12 tests a more dispersed count.

In [ ]:
by_year = defaultdict(int)
for x in eligible:
    by_year[x["year"]] += 1
# Expected number of complete claims each year if the book were at 2026's size and prices and the cut were T:
# the count, grossed up for late reports, moved from the year's own cut to T with the fitted alpha, and scaled by exposure.
cut_factor = {y: trend_to_treaty(y) ** ALPHA for y in YEARS}
exposure_ratio = {y: onlevel[y] / ONLEVEL_2026 for y in YEARS}
adjusted = {y: by_year[y] * count_factor[y] * cut_factor[y] / exposure_ratio[y] for y in YEARS}
print(f"{'year':>6s} {'claims':>7s} {'late reports':>13s} {'to cut T':>9s} {'exposure':>9s} {'claims a year at 2026 size':>27s}")
for y in YEARS:
    print(f"{y:>6d} {by_year[y]:>7d} {count_factor[y]:>13.3f} {cut_factor[y]:>9.3f} {exposure_ratio[y]:>9.3f} {adjusted[y]:>27.1f}")

# A Poisson GLM on the counts: the offset is the expected count if nothing but exposure, cut and late reports varied.
offset = [math.log(exposure_ratio[y] / (count_factor[y] * cut_factor[y])) for y in YEARS]
data = {"year": [float(y - 2020) for y in YEARS]}
level_fit = Glm("poisson", "log", dispersion="pearson").fit(
    Terms().intercept().fit(data).design(data, offset=offset), [float(by_year[y]) for y in YEARS])
trend_fit = Glm("poisson", "log", dispersion="pearson").fit(
    Terms().intercept().numeric("year").fit(data).design(data, offset=offset), [float(by_year[y]) for y in YEARS])
LAMBDA = math.exp(level_fit.coefficients[0])
DISPERSION = max(1.0, level_fit.dispersion)
print(f"\nclaims a year over {money(T)} at 2026 size and prices: {LAMBDA:.1f}; Pearson dispersion {level_fit.dispersion:.2f}")
print(f"residual trend in the counts: {math.exp(trend_fit.coefficients[1]) - 1:+.1%} a year "
      f"(standard error {trend_fit.std_errors[1]:.3f} on the log scale, p-value {trend_fit.p_values[1]:.2f})")
frequency = claim_count(LAMBDA, DISPERSION)
print(f"count model: {frequency}")
first = YEARS[0]
expected_first = LAMBDA * exposure_ratio[first] / (count_factor[first] * cut_factor[first])
p_low = sum(math.exp(-expected_first) * expected_first**k / math.factorial(k) for k in range(by_year[first] + 1))
print(f"{first}: {expected_first:.1f} complete claims expected at the fitted level, {by_year[first]} seen "
      f"(a Poisson count is that low or lower with probability {p_low:.0%})")

Three remarks on the counts:

* The residual trend is small and far from significant (a p-value over 0.3), so we do not apply it.
* 2016 has the lowest adjusted count. The fitted level expects about 6 complete claims in that year, and there were 3. A Poisson count gives this result about one year in seven, so the year stays in.
* The move from the cut of a year to `T` assumes that each risk can produce a claim above that cut. A risk insured for less than the cut cannot. The adjustment is thus a little too small for the oldest years. By the generator's mix of sums insured, it is about 3% too small on the 2016 count, and less than 1% from 2023. We do not change it.

## 6. Experience rating

**Experience rating** prices a layer from the cedant's own claims. There are two estimates:

* The **as-if burning cost**. Put the trended, developed large claims of each year through the current layer, scale to 2026 exposure, and average the years. It makes no assumption about the tail, and it has a lot of noise for high layers. A year with no claim over the attachment shows nothing, whatever the risk of the layer. `PreparedLosses.burning_cost` does this calculation. It puts the trended claims of each year through the layers. It grosses up the layer total of each year by the count factor for late reports (section 4) and divides it by the on-level premium of the year. That ratio times the 2026 on-level premium is the cost of the year at 2026 size.
* A **model**. The fitted Pareto for severity and the count for frequency go into a `CollectiveModel`. This gives the expected loss of the layer in closed form from all the claims together. The policy limit of its risk caps each claim, so the severity is a mixture of Pareto distributions capped at different limits. `PolicyLimits` is that mixture, with the as-if limits of the claims in the data. `limited_severity` puts it on a grid with `Grid.rounding`, which keeps the mass at each limit (the claims that the limit caps) on the limit.

`occurrence_layer_statistics` adds the shape: how often a claim reaches the layer or uses all of it.

In [ ]:
STEP = 25_000.0
LAYERS = [(t["limit"], t["attachment"]) for t in terms]
NAMES = [t["name"] for t in terms]
MAX_LOSS = float(profile_rows[-1]["upper_sum_insured"])  # the largest sum insured in the book


def limited_severity(alpha, policy_limits, threshold=T, step=STEP):
    """The severity of a claim over the threshold on a grid: Pareto(threshold, alpha) cut at the policy limit of the
    risk, with the limits taken as the claims' own as-if limits, each with equal weight."""
    capped = PolicyLimits(Pareto(threshold, alpha), policy_limits, [1.0] * len(policy_limits))
    grid, _ = Grid.rounding(capped, step, int(max(policy_limits) / step) + 2)
    return grid


policy_limits = [x["limit"] * x["trend"] for x in eligible]
severity = limited_severity(ALPHA, policy_limits)
experience = CollectiveModel(frequency, severity)
experience_loss = [experience.layer_mean(limit, att) for limit, att in LAYERS]

# As-if burning cost: each year's trended claims through the layers (per claim, no annual terms), grossed up for
# late reports by the count pattern at the year's average age, over the year's on-level premium. The claims under
# the analysis threshold stay in the record: they are far below the lowest attachment and add nothing.
layers_only = Tower([Layer(t["name"], t["limit"], t["attachment"]) for t in terms])
costs = prepared.burning_cost(layers_only, YEARS, premium=[onlevel[y] for y in YEARS], development=count_pattern)
by_layer = costs.to_dict()
# A loss cost times the 2026 on-level premium is the cost at 2026 size; the mean weights the years equally.
burning = {n: {y: c[j] * ONLEVEL_2026 for y, c in zip(by_layer["year"], by_layer["loss_cost"])}
           for j, n in enumerate(NAMES)}
reaching = {name: sum(x["loss"] > att for x in eligible) for name, (limit, att) in zip(NAMES, LAYERS)}
print(f"{'year':>6s} " + " ".join(f"{n:>11s}" for n in NAMES))
for y in YEARS:
    print(f"{y:>6d} " + " ".join(f"{money(burning[n][y]):>11s}" for n in NAMES))
burning_cost = [m * ONLEVEL_2026 for m in costs.mean_loss_cost]
print(f"{'mean':>6s} " + " ".join(f"{money(b):>11s}" for b in burning_cost))
print(f"{'model':>6s} " + " ".join(f"{money(m):>11s}" for m in experience_loss))
print(f"claims reaching the layer in ten years: " + ", ".join(f"{n} {reaching[n]}" for n in NAMES))

for name, (limit, att) in zip(NAMES, LAYERS):
    s = occurrence_layer_statistics(frequency, severity, 1.0, limit, att)
    print(f"{name:>11s}: of the claims over {money(T)}, {s.pr_attach:.1%} reach the layer and {s.pr_detach:.1%} "
          f"use all of its limit; {s.count.mean:.2f} losses to the layer a year")

## 7. Exposure rating

**Exposure rating** prices a layer from the cedant's portfolio, not from its claims. The risk profile lists bands of sum insured with their premium. An **exposure curve** gives the share of the expected loss of a risk below a given fraction of its sum insured. The part above the attachment of the layer, within its limit, is the share of the layer. Apply the curve to the ground-up expected loss of each band (premium times the loss ratio from section 3). The result is the expected loss of the layer. It uses the claim history only through that one loss ratio.

`RiskProfile` does this with an MBBEFD exposure curve. The Swiss Re curves are the one-parameter family `Mbbefd.swiss_re(c)`. A lower `c` puts more of the loss in the large claims. The choice of curve is the main judgement of this method. `Mbbefd.swiss_re_for_mean_rate` and `Mbbefd.swiss_re_for_layer_share` find `c` from a mean destruction rate or from the share of one layer in the loss of a risk. The submission gives neither, so the table shows three curves, and we choose one.

In [ ]:
lower = [float(r["lower_sum_insured"]) for r in profile_rows]
upper = [float(r["upper_sum_insured"]) for r in profile_rows]
risks = [float(r["risks"]) for r in profile_rows]
mean_si = [float(r["total_sum_insured"]) / float(r["risks"]) for r in profile_rows]
band_premium = [float(r["premium"]) for r in profile_rows]


def exposure_rated(c, loss_ratio=ELR):
    profile = RiskProfile(mean_si, risks, Mbbefd.swiss_re(c), premiums=band_premium, loss_ratio=loss_ratio,
                          lower=lower, upper=upper, spread="tilted")
    return [profile.expected_layer_loss(limit, att) for limit, att in LAYERS]


print(f"ground-up expected loss {money(ELR * profile_premium)} = {ELR:.1%} of the profile's premium {money(profile_premium)}")
print(f"{'curve':>14s} " + " ".join(f"{n:>11s}" for n in NAMES))
for c in (2.0, 3.0, 4.0):
    print(f"{'Swiss Re c=' + format(c, 'g'):>14s} " + " ".join(f"{money(m):>11s}" for m in exposure_rated(c)))
CURVE = 3.0  # industrial and commercial property
exposure_loss = exposure_rated(CURVE)
print(f"\nselected c={CURVE:g}: " + ", ".join(f"{n} {money(m)}" for n, m in zip(NAMES, exposure_loss)))

## 8. Blend experience and exposure by credibility

Do not use either estimate alone. Experience shows the cedant's own history, but it is thin for high layers. Exposure rating is stable, but it depends on a curve chosen by judgement. **Credibility** weights them: `Z` times experience plus `(1 - Z)` times exposure.

`LimitedFluctuation` (classical credibility) gives the number of claims that the experience needs for its estimate to be within `k` of the truth with probability `p`. For the pure premium of a layer (frequency times severity), that number increases with `1 + cv**2`. Here `cv` is the coefficient of variation of the loss of the layer per claim over `T`. Few claims reach a layer, so its `cv` is large, and the standard is far above the number of claims that we have. `Z` is the square root of the ratio. Layer losses are volatile, so we use a tolerance `k` of 20%, not the 5% of primary ratemaking. `Z` is lower for higher layers.

This standard comes from primary ratemaking. The `cv` here is of the loss of the layer per claim over `T`, zeros included, to go with the count of claims over `T`. `layer_cv(severity, limit, attachment)` gives the `cv` per claim that reaches the layer, to go with the count of those claims. The last line below shows it.

The standard measures the volatility of an *empirical* estimate: the burning cost, from the few claims that reach the layer. But the experience estimate in this blend is the fitted model of section 6. That model learns the tail from all the claims over `T`, so it is more stable than a burning cost. Its uncertainty is that of its parameters, which section 12 draws. Thus the `Z` below is a conservative weight for the model. Section 12 shows the effect on the price when `Z` is three times larger.

In [ ]:
CRED_P, CRED_K = 0.90, 0.20  # within 20% of the mean with probability 90%
lf = LimitedFluctuation(p=CRED_P, k=CRED_K)


def blend_layers(sev, experience_loss, exposure_loss, n_claims, scale=1.0):
    """Credibility-weight experience and exposure layer by layer. The standard is for the layer's pure premium
    (frequency and severity both vary), in claims over T; the cv is that of the layer's loss per claim over T.
    `scale` multiplies the credibility (capped at 1), for the sensitivity of section 12."""
    blend, credibility, cvs = [], [], []
    for (limit, att), e, x in zip(LAYERS, experience_loss, exposure_loss):
        per_claim, _ = sev.map(lambda v: min(max(v - att, 0.0), limit))
        cv = math.sqrt(per_claim.variance()) / per_claim.mean()
        z = min(1.0, scale * lf.credibility(n_claims, lf.pure_premium_standard(cv)))
        blend.append(credibility_weighted(e, z, x))
        credibility.append(z)
        cvs.append(cv)
    return blend, credibility, cvs


blended, weights, cvs = blend_layers(severity, experience_loss, exposure_loss, len(eligible))
print(f"full-credibility standard for a frequency within {CRED_K:.0%} with probability {CRED_P:.0%}: "
      f"{lf.frequency_standard():.0f} claims; for a layer's pure premium it is that times (1 + cv squared)")
print(f"{'layer':>11s} {'cv of layer loss':>17s} {'claims':>7s} {'credibility':>12s} {'experience':>11s} {'exposure':>9s} {'blend':>9s}")
for name, cv, z, e, x, b in zip(NAMES, cvs, weights, experience_loss, exposure_loss, blended):
    print(f"{name:>11s} {cv:>17.2f} {len(eligible):>7d} {z:>12.2f} {money(e):>11s} {money(x):>9s} {money(b):>9s}")
print("cv per claim reaching the layer (layer_cv): "
      + ", ".join(f"{n} {layer_cv(severity, limit, att):.2f}" for n, (limit, att) in zip(NAMES, LAYERS)))

This standard gives the experience a small credibility, and the credibility decreases with the layer. The claims data still matters: it sets the loss ratio of section 3 and the size of the book in section 5. Section 12 shows what moves the price. But with these weights, the choice of exposure curve has more effect than the tail fitted from a hundred claims. The weights are conservative for a fitted model, so section 12 also multiplies them by three.

## 9. The loss model and the treaty terms

To get the distribution of the annual payment of each layer, and not only its mean, we need one model for the whole program. `fit_references` fits a piecewise Pareto severity that gives the three blended layer losses. This also sets the number of claims over the lowest attachment.

Be careful with the matching. It returns a model for all inputs that permit one, and the pieces between the attachments can have very large or very small alphas. `diagnostics()` lists each alpha below 1 and each change by more than a factor of 2 between adjacent pieces. `fit_references(..., alpha_bounds=(lo, hi))` keeps the alphas in a range. The loss to each layer is exact. The shape between the attachments is not data. Read the alphas and the warnings.

Then `Tower.on_grid` applies the terms: the limit and attachment of each layer, the paid reinstatements, and the annual limit that the reinstatements give. The reinstatements are charged at the rate in the terms, pro rata as to amount. The gross loss cannot be more than the largest sum insured. The result is an exact annual distribution for each layer, with no simulation error.

In [ ]:
# The blended layer losses fix a piecewise Pareto above the lowest attachment, and with it the number of claims
# over that attachment (exposure rating has no claim count, so the matching derives one).
EXCESS_FREQUENCY = experience.excess_frequency(LAYERS[0][1])
model = fit_references([(limit, att, el) for (limit, att), el in zip(LAYERS, blended)], default_alpha=ALPHA)
alphas = model.severity.alpha
print(f"matched model: {model.frequency:.3f} claims a year over {money(LAYERS[0][1])} (the experience model has "
      f"{EXCESS_FREQUENCY:.3f}); alphas of its pieces {', '.join(f'{a:.2f}' for a in alphas)}")
diagnostics = model.diagnostics()
print(f"diagnostics: alphas from {diagnostics['min_alpha']:.2f} to {diagnostics['max_alpha']:.2f}, "
      f"{len(diagnostics['warnings'])} warnings (alphas below 1, jumps of more than a factor of 2 between pieces)")
print("(a piece with a very large or very small alpha is where the three layer losses and the frequency "
      "disagree; the loss to each layer is exact, the shape between attachments is not)")
print(f"{'layer':>11s} {'blended':>9s} {'model':>9s}")
for name, (limit, att), el in zip(NAMES, LAYERS, blended):
    print(f"{name:>11s} {money(el):>9s} {money(model.layer_loss(limit, att)):>9s}")

POINTS = 2**15
# No claim can exceed the largest sum insured: Grid.rounding puts the mass beyond it on the last point.
loss_grid, _ = Grid.rounding(model.severity, STEP, int(MAX_LOSS / STEP) + 1)
loss_frequency = claim_count(model.frequency, 1.0)


def program(deposits):
    """The proposed layers with their reinstatements; `deposits` are the premiums the reinstatements are charged on."""
    return Tower([Layer(t["name"], t["limit"], t["attachment"], premium=d,
                        reinstatement_rates=[t["reinstatement_rate"]] * t["reinstatements"])
                  for t, d in zip(terms, deposits)])


# One unit of deposit premium on every layer: the expected reinstatement premium is then per unit of deposit.
unit_grids = program([1.0] * len(terms)).on_grid(loss_frequency, loss_grid, POINTS)
grids = unit_grids
reinstatement_load = list(unit_grids.expected_reinstatement_premium)
print(f"\non the grid: {grids.on_points=}, annual expected loss "
      + ", ".join(f"{n} {money(g.mean())}" for n, g in zip(NAMES, grids.ceded)))
print("expected reinstatement premium per unit of deposit: " + ", ".join(f"{n} {r:.3f}" for n, r in zip(NAMES, reinstatement_load)))

Terms that depend on the time in the year (a reinstatement pro rata as to time, an event limit) need simulated events. `simulate_events` and `Tower.apply` apply the same program to simulated years. Here they only confirm the grid. The means agree within the simulation error, which is larger for the top layer, because few simulated years pay anything.

In [ ]:
from prospicio.aggregate import simulate_events

# The same program by simulation: terms that are not on the grid (a reinstatement pro rata as to time, an event
# limit) need events. Here the check is that the two agree.
events = simulate_events(loss_frequency, loss_grid, N_SIMS, 11)
simulated = program([1e6] * len(terms)).apply(events)
print(f"{'layer':>11s} {'grid':>9s} {'simulated':>10s} {'difference':>11s}")
for name, g in zip(NAMES, grids.ceded):
    s = simulated.marginal(["ceded", name]).mean()
    print(f"{name:>11s} {money(g.mean()):>9s} {money(s):>10s} {s / g.mean() - 1:>+11.1%}")

## 10. Risk metrics for each layer

The expected loss of a layer is not sufficient. How often losses reach it and how bad a bad year is set the capital behind it. The annual distribution of each layer gives these values:

* the probability that it pays anything in the year (attachment probability).
* the probability that a single loss uses all its limit (exhaustion).
* the probability that the year uses the whole annual limit.
* the standard deviation.
* the 99% **value at risk** (VaR, the loss exceeded once in a hundred years).
* the 99% **tail value at risk** (TVaR, the average of the worst 1% of years).

The largest single loss of a year, from the simulated events, is the occurrence exceedance that a cat modeller quotes as a return period.

The blended layer losses set only the mean of each layer. All other figures in this table (the probabilities, the standard deviation, VaR and TVaR) also depend on the shape of the matched severity between the attachments. Section 9 says that this shape is not data. Section 12 compares it with a smooth shape.

In [ ]:
TAIL = Distortion.tvar(0.99)
metrics = {}
print(f"{'layer':>11s} {'P(any loss)':>12s} {'P(limit used)':>14s} {'P(annual limit)':>16s} {'sd':>8s} {'VaR 99%':>9s} {'TVaR 99%':>9s}")
for t, g in zip(terms, grids.ceded):
    annual_limit = t["limit"] * (1 + t["reinstatements"])
    m = {"any": 1 - g.cdf(0.0), "limit": 1 - g.cdf(t["limit"] - STEP / 2), "annual": 1 - g.cdf(annual_limit - STEP / 2),
         "sd": math.sqrt(g.variance()), "var": g.quantile(0.99), "tvar": TAIL.measure(g)}
    metrics[t["name"]] = m
    print(f"{t['name']:>11s} {m['any']:>12.1%} {m['limit']:>14.1%} {m['annual']:>16.2%} {money(m['sd']):>8s} "
          f"{money(m['var']):>9s} {money(m['tvar']):>9s}")

# The largest single loss of a year (occurrence exceedance), from the simulated events, against the layers.
largest = events.maxima()
print("\nlargest claim of the year, return periods: " + ", ".join(
    f"1 in {rp}: {money(largest.quantile(1 - 1 / rp))}" for rp in (5, 10, 25, 50, 100)))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.8))
xs = [i * 0.05e6 for i in range(0, 501)]
for name, g in zip(NAMES, grids.ceded):
    ax.plot([x / 1e6 for x in xs], [1.0 - g.cdf(x) for x in xs], label=name)
ax.set_yscale("log")
ax.set_ylim(1e-3, 1.0)
ax.set_xlim(0, 25)
ax.set_xlabel("annual loss to the layer (millions)")
ax.set_ylabel("probability of exceeding")
ax.set_title("Annual loss to each layer, on the grid")
ax.legend()
fig.tight_layout()
plt.show()

## 11. Technical premium and the broker's quotes

The **technical premium** is the price that the reinsurer's own model supports. As in the guide, it has three steps:

1. **Expected loss** from the annual distribution.
2. **Cost of capital**. The capital is the 99% TVaR of the annual loss of the layer (the capital standard) less the premium received. The reinsurer asks for a return of 10% on it. `price(grid, TAIL, cost_of_capital=RETURN)` gives the risk premium from the annual loss of the layer, its assets at that standard, and the return. The TVaR is of the gross loss. In the years that use the limit, the reinsurer also receives reinstatement premium. Thus this standard is conservative by that premium, most for the 5m xs 5m layer with its two reinstatements.
3. **Expenses and brokerage** are shares of the premium (3%, and the broker's 10%). Thus we divide the risk premium by one minus both. `Quote.technical` does that gross-up and gives the ratios that the market quotes. `price_layers` below does the three steps for each layer.

These values are the reinsurer's own policy: the 10% target return, the 3% of expenses, the line sizes of section 13, and the 6% floor. Below the floor, a contract is not worth writing at any size. The submission does not give them. Replace them with the values of your company.

The quotes are on the **deposit** (the premium at inception). The reinstatement premium comes in addition, in the years that losses use the limit. One unit of deposit brings a known expected reinstatement premium (section 9). Thus the technical deposit is the technical premium divided by one plus that load.

We also compare with the quote as a **return at the quote**. This is the premium that the reinsurer keeps after expenses and brokerage, against the same loss and the same assets (the capital is the assets less that premium). `Quote.at` gives it for a quoted premium.

The technical premium here is for one period. It does not include the investment income on the premium held until the claims are paid, or the cost of holding capital until then. These two effects are in opposite directions, and a one-period price is the usual start for a property layer. When the payment tail matters, use `PaymentPattern.from_development` with `price_over_payout` to restate the price over the payout. Give `PaymentPattern.from_development` the factors to ultimate of the layer's own paid losses, by age in years from inception. The paid chain ladder of section 3 is not that pattern. It is for the whole book from the ground up, by age in months. Losses to an excess layer are paid more slowly than the book, so that pattern understates the duration of the layer.

A stand-alone capital standard is conservative for the high layers, where the 99% TVaR approaches the full limit (section 10). A reinsurer with a diversified book can accept less. The second cross-check below prices all layers with one price of risk.

In [ ]:
RETURN = 0.10  # the return on capital the reinsurer asks for
EXPENSES = 0.03  # internal expenses, a share of premium
FLOOR_RETURN = 0.06  # below this the contract is not worth writing at any size
TOLERANCE = 0.05  # a quote within 5% of a price is too close to call, given the estimation error of section 12


def price_layers(grids, loads):
    """Technical premium of each layer from its annual loss: risk premium at the 99% TVaR capital standard and the
    required return, then grossed up for expenses and brokerage; and the deposit that, with the reinstatement
    premium it brings, equals it."""
    rows = []
    for t, g, load in zip(terms, grids.ceded, loads):
        quote = Quote.technical(price(g, TAIL, cost_of_capital=RETURN), expenses=EXPENSES, brokerage=BROKERAGE,
                                limit=t["limit"])
        deposit = quote.premium / (1 + load)
        rows.append({"name": t["name"], "loss": quote.expected_loss, "capital": quote.capital,
                     "risk": quote.risk_premium, "technical": quote.premium, "deposit": deposit,
                     "rol": deposit / t["limit"], "assets": quote.assets, "quote": quote})
    return rows


def at_quote(row, t, load):
    """What the broker's quote earns the reinsurer: the quoted deposit with the reinstatement premium it brings, and
    the return of the premium left after expenses and brokerage, on the same loss and assets."""
    total = t["quoted_rol"] * t["limit"] * (1 + load)
    return total, row["quote"].at(total).return_on_capital


priced = price_layers(grids, reinstatement_load)
print(f"{'':11s} {'expected':>9s} {'capital':>8s} {'risk':>8s} {'technical':>10s} {'deposit':>9s} {'tech. ROL':>10s} {'quoted ROL':>11s} "
      f"{'multiple':>9s} {'return at quote':>16s}")
for row, t, load in zip(priced, terms, reinstatement_load):
    total, ret = at_quote(row, t, load)
    row.update(quoted_total=total, quoted_return=ret)
    print(f"{row['name']:>11s} {money(row['loss']):>9s} {money(row['capital']):>8s} {money(row['risk']):>8s} "
          f"{money(row['technical']):>10s} {money(row['deposit']):>9s} {row['rol']:>10.1%} {t['quoted_rol']:>11.1%} "
          f"{row['quote'].multiple:>9.2f} {ret:>16.1%}")

**Cross-checks.** There are two:

1. Calibrate a proportional hazard distortion to the quote of each layer. A distortion weights the tail of a loss distribution, so that bad years count for more than their probability. The weight on a 1% year shows how much the quote charges for risk. Quotes that charge about the same weight in each layer are consistent with each other.
2. Take the price of risk that makes layer 1 equal to its technical premium, and apply it to the other layers. This is a market-style price. It charges each layer for its risk at one rate, not for a stand-alone capital that increases with the limit of the layer.

A quote clearly above the one-price premium (by more than 5%, our tolerance for estimation error) supports a smaller line, or a referral to the portfolio view. This applies also where the stand-alone return is too low.

In [ ]:
# Two cross-checks of the quote. First, does it hang together as one price of risk? Calibrate a proportional hazard
# distortion to each layer's quote (net of expenses and brokerage); it weights a 1% year and a 10% year more than
# their probability by an amount that says how much risk is being charged for.
print(f"{'layer':>11s} {'weight on a 1% year':>20s} {'weight on a 10% year':>21s}")
for row, g in zip(priced, grids.ceded):
    implied = calibrate("ph", g, row["quote"].at(row["quoted_total"]).risk_premium)
    print(f"{row['name']:>11s} {implied.g(0.01):>20.1%} {implied.g(0.10):>21.1%}")

# Second, the price of every layer under the price of risk of the first (calibrated to its risk premium), the
# approach of the guide's section 4: one distortion for the program.
one_price = calibrate("ph", grids.ceded[0], priced[0]["risk"])
consistent = [Quote.technical(price(g, TAIL, distortion=one_price), expenses=EXPENSES, brokerage=BROKERAGE).premium
              for g in grids.ceded]
print(f"\n{'layer':>11s} {'technical (TVaR)':>17s} {'one price of risk':>18s} {'quoted':>9s} {'quoted / one price':>19s}")
for row, c in zip(priced, consistent):
    row["consistent"] = c
    print(f"{row['name']:>11s} {money(row['technical']):>17s} {money(c):>18s} {money(row['quoted_total']):>9s} "
          f"{row['quoted_total'] / c - 1:>+19.1%}")

**Why the two prices differ for the high layers.** The first table shows that the broker's three quotes charge about the same weight on a bad year. Thus they are consistent as one price of risk. The second table shows where our technical premium is relative to that price.

* For 5m xs 5m, the two prices agree by construction.
* For 20m xs 20m, the technical premium (2.19m) is 2.3 times the one-price premium (0.94m). Its capital standard is the 99% TVaR of 18.66m on a 20m limit. Thus the stand-alone capital (16.75m) is almost the whole limit, for a layer that pays in about one year in thirty. One price of risk charges that layer for its tail at the rate of the working layer. This is nearer to how a reinsurer that holds many such layers carries it. The broker's 1.21m is 29% above that price. Thus the layer is not underpriced as a risk, but it does not give a sufficient return on stand-alone capital.
* For 10m xs 10m, the quote is 2% above the one-price premium. This is inside the 5% tolerance, so neither price clearly supports it.

## 12. Sensitivity and parameter uncertainty

Each figure so far comes from estimates. `evaluate` does sections 3 to 11 as one function, so we can change inputs and calculate the premiums again from the listing. The first table changes one input at a time, from the loss ratio of section 3 to the credibility of section 8. The second table draws the Pareto alpha and the claim frequency from their sampling distributions, and the exposure curve parameter between 2 and 4. It runs the whole calculation for each draw. The curve range is wide on purpose: it represents the judgement that no data in the submission can settle.

In [ ]:
def evaluate(trend=SEVERITY_TREND, alpha=None, frequency_scale=1.0, dispersion=1.0, curve=CURVE,
             threshold=ANALYSIS_THRESHOLD, elr_shift=0.0, credibility_scale=1.0):
    """Steps 3 to 11 as one function, so an input can be moved and the technical premiums recomputed:
    the Cape Cod loss ratio (moved by `elr_shift`) and the large losses at `trend`, the complete claims over
    `threshold`, the Pareto fit (or `alpha` given), the claim count scaled by `frequency_scale`, experience and
    exposure layer losses, their credibility blend (credibility times `credibility_scale`), the matched model,
    the program on the grid and the technical premium of each layer."""
    def factor(date):  # the same trend as `LargeLossExperience.prepare` in section 4, at `trend`
        return trend_factor(trend, TREATY_MIDDLE - date)

    elr = loss_ratio(CapeCod(trend=trend).fit(triangle, "incurred", "onlevel").ultimate, trend) + elr_shift
    complete = [x for x in large if x["ultimate"] >= threshold]
    fit = Pareto.fit([x["ultimate"] * factor(claims[x["id"]]["accident"]) for x in complete], threshold,
                     reporting_thresholds=[threshold * factor(claims[x["id"]]["accident"]) for x in complete],
                     censored=[x["at_limit"] for x in complete]).alpha if alpha is None else alpha
    # The Poisson maximum likelihood level of the adjusted counts: total claims over total expected-per-unit.
    per_unit = sum(exposure_ratio[y] / (count_factor[y] * factor(y + 0.5) ** fit) for y in YEARS)
    lam = frequency_scale * len(complete) / per_unit
    sev = limited_severity(fit, [x["limit"] * factor(claims[x["id"]]["accident"]) for x in complete], threshold)
    exp_model = CollectiveModel(claim_count(lam, dispersion), sev)
    experience_ = [exp_model.layer_mean(limit, att) for limit, att in LAYERS]
    blend, credibility, _ = blend_layers(sev, experience_, exposure_rated(curve, elr), len(complete),
                                         credibility_scale)
    matched = fit_references([(limit, att, el) for (limit, att), el in zip(LAYERS, blend)], default_alpha=fit)
    grid, _ = Grid.rounding(matched.severity, STEP, int(MAX_LOSS / STEP) + 1)
    annual = program([1.0] * len(terms)).on_grid(claim_count(matched.frequency, dispersion), grid, POINTS)
    rows = price_layers(annual, list(annual.expected_reinstatement_premium))
    for row, t, load in zip(rows, terms, annual.expected_reinstatement_premium):
        row["quoted_total"], row["quoted_return"] = at_quote(row, t, load)
    return {"alpha": fit, "lambda": lam, "elr": elr, "experience": experience_, "blend": blend, "rows": rows}


base = evaluate()
print("the function reproduces the step-by-step result: "
      + ", ".join(f"{r['name']} {r['technical']:,.0f} against {p['technical']:,.0f}" for r, p in zip(base["rows"], priced)))

In [ ]:
ELR_SE = spread / math.sqrt(len(YEARS))  # standard error of the ten-year loss ratio, section 3
scenarios = [("base case", {}),
             (f"loss ratio {2 * ELR_SE:.1%} lower", {"elr_shift": -2 * ELR_SE}),
             (f"loss ratio {2 * ELR_SE:.1%} higher", {"elr_shift": 2 * ELR_SE}),
             ("analysis threshold 1m", {"threshold": 1e6}),
             ("experience credibility x3", {"credibility_scale": 3.0}),
             ("tail alpha 0.25 lower (heavier)", {"alpha": ALPHA - 0.25}),
             ("tail alpha 0.25 higher (lighter)", {"alpha": ALPHA + 0.25}),
             ("cost trend 1.5 points lower", {"trend": SEVERITY_TREND - 0.015}),
             ("cost trend 1.5 points higher", {"trend": SEVERITY_TREND + 0.015}),
             ("claim frequency 20% lower", {"frequency_scale": 0.8}),
             ("claim frequency 20% higher", {"frequency_scale": 1.2}),
             ("claims cluster (count dispersion 3)", {"dispersion": 3.0}),
             ("exposure curve c=2 (heavier)", {"curve": 2.0}),
             ("exposure curve c=4 (lighter)", {"curve": 4.0})]
results = {label: (base if not kw else evaluate(**kw)) for label, kw in scenarios}
print(f"{'technical rate on line':>38s}  " + "  ".join(f"{n:>11s}" for n in NAMES) + "   " + "  ".join(f"{'return ' + n.split()[0]:>10s}" for n in NAMES))
for label, res in results.items():
    print(f"{label:>38s}  " + "  ".join(f"{r['rol']:>11.1%}" for r in res["rows"]) + "   "
          + "  ".join(f"{r['quoted_return']:>10.1%}" for r in res["rows"]))

Read the table down a column. An input that changes the technical rate on line by a small amount needs no more work. The inputs that change it most are where to question the submission.

In this case, two inputs change the rate on line of the working layer by about six points:

* The exposure curve: c=2 and c=4 give 47.1% and 35.1%, against 41.1%.
* The clustering of claims: a count dispersion of 3 gives 47.5%, and decreases the return at the quote from 11.3% to 7.9%.

The dispersion is the more difficult of the two to settle. The Pearson estimate from ten yearly counts is 0.80, with a floor at 1. Ten counts cannot show the difference between a dispersion of 1 and one of 2 or 3, so it is a judgement, as the curve is. Each of the other inputs changes the rate on line by less than two points. These inputs are the tail alpha, the frequency, the cost trend, the loss ratio (moved by two standard errors), the analysis threshold, and credibility three times larger. This result follows from the low credibility in section 8.

The top layer changes very little with any input except the curve. Its premium is mostly the capital charge on a 99% TVaR near its whole limit. Thus its price comes more from the choice of capital standard than from the loss model.

**The shape of the matched model.** The matching of section 9 sets the mean of each layer but not the shape between the attachments. The probabilities, VaR, TVaR and thus the capital of sections 10 and 11 depend on that shape. To measure this effect, price each layer again with a smooth shape. This is the severity of the experience model: one Pareto alpha over `T`, capped at the policy limits. For each layer, its claim frequency is scaled so that the expected loss of the layer is still the blended one.

In [ ]:
# Each layer on its own grid: the experience severity with the frequency that gives the blended layer loss.
smooth = []
for i, (t, el, e) in enumerate(zip(terms, blended, experience_loss)):
    layer_grids = program([1.0] * len(terms)).on_grid(claim_count(LAMBDA * el / e, 1.0), severity, POINTS)
    smooth.append((layer_grids.ceded[i], layer_grids.expected_reinstatement_premium[i]))
smooth_rows = price_layers(type("SmoothGrids", (), {"ceded": [g for g, _ in smooth]}),
                           [load for _, load in smooth])
print(f"{'':11s} {'P(any loss)':>23s} {'TVaR 99%':>21s} {'technical ROL':>21s}")
print(f"{'layer':>11s} {'matched':>11s} {'smooth':>11s} {'matched':>10s} {'smooth':>10s} {'matched':>10s} {'smooth':>10s}")
for t, g, (sg, _), row, srow in zip(terms, grids.ceded, smooth, priced, smooth_rows):
    print(f"{t['name']:>11s} {1 - g.cdf(0.0):>11.1%} {1 - sg.cdf(0.0):>11.1%} {money(TAIL.measure(g)):>10s} "
          f"{money(TAIL.measure(sg)):>10s} {row['rol']:>10.1%} {srow['rol']:>10.1%}")

The smooth shape changes the technical rates on line of the two lower layers by less than half a point. But losses reach the middle layer in fewer years (11.9% against 15.4%). The top layer changes most. Its TVaR decreases from 18.66m to 15.86m, and its rate on line from 10.8% to 9.4%, still well above the broker's 6%. The matched shape matters for the capital of the top layer, not for the decision.

In [ ]:
# Parameter uncertainty: draw the Pareto alpha from its sampling distribution (normal, standard error alpha over
# the square root of the uncensored claims) and the claim frequency from a gamma with as many claims as were
# observed (fewer if the counts are over-dispersed), and the exposure curve parameter c uniformly between 2 and 4,
# re-run the chain, and look at the spread of the premium.
import random

N_DRAWS = 200 if N_SIMS >= 20_000 else 20  # fewer when PROSPICIO_EXAMPLE_SIMS is lowered, as in CI
rng = random.Random(2026)
effective_claims = len(eligible) / max(1.0, level_fit.dispersion)
draws = []
for _ in range(N_DRAWS):
    a = max(1.15, rng.gauss(ALPHA, ALPHA / math.sqrt(n_uncensored)))
    f = rng.gammavariate(effective_claims, 1 / effective_claims)
    draws.append(evaluate(alpha=a, frequency_scale=f, curve=rng.uniform(2.0, 4.0)))


def percentile(values, q):
    v = sorted(values)
    return v[min(len(v) - 1, int(q * len(v)))]


print(f"{N_DRAWS} draws of alpha (standard error {ALPHA / math.sqrt(n_uncensored):.2f}), of the claim frequency "
      f"({1 / math.sqrt(effective_claims):.0%} sampling error) and of the exposure curve")
print(f"{'layer':>11s} {'technical ROL, 5% / 50% / 95%':>32s} {'return at quote, 5% / 50% / 95%':>34s} {'P(quote covers technical)':>26s}")
cover, return_band, rol_band = [], [], []
for i, (t, row) in enumerate(zip(terms, priced)):
    rol = [d["rows"][i]["rol"] for d in draws]
    ret = [d["rows"][i]["quoted_return"] for d in draws]
    cover.append(sum(t["quoted_rol"] >= r for r in rol) / len(rol))
    return_band.append((percentile(ret, 0.05), percentile(ret, 0.95)))
    rol_band.append((percentile(rol, 0.05), percentile(rol, 0.95)))
    print(f"{t['name']:>11s} {percentile(rol, 0.05):>10.1%} {percentile(rol, 0.5):>9.1%} {percentile(rol, 0.95):>9.1%} "
          f"{percentile(ret, 0.05):>13.1%} {percentile(ret, 0.5):>9.1%} {percentile(ret, 0.95):>9.1%} {cover[-1]:>22.0%}")

The last column gives the share of draws in which the broker's rate on line is at least our technical rate on line. Read it with three cautions:

* The 5% and 95% points of a few hundred draws are approximate.
* The loss ratio, the cost trend, the credibility weights and the capital standard stay at their selected values. Thus the band is narrower than the whole uncertainty. The first table changes those inputs one at a time.
* The exposure curve is drawn uniformly between c=2 and c=4 around the selected c=3. This range is a judgement. A narrower range moves the probability towards 0 or 1.

In `evaluate`, the frequency of each draw is fitted again with the alpha of that draw. Thus the two are not independent in their effect on the layers.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.8))
for i, (t, row) in enumerate(zip(terms, priced)):
    rol = [d["rows"][i]["rol"] for d in draws]
    ax.plot([i, i], [percentile(rol, 0.05), percentile(rol, 0.95)], color="C0", lw=6, alpha=0.4,
            label="technical, 5% to 95% of the draws" if i == 0 else None)
    ax.plot(i, row["rol"], "o", color="C0", label="technical, central" if i == 0 else None)
    ax.plot(i, t["quoted_rol"], "D", color="C3", label="broker's quote" if i == 0 else None)
ax.set_xticks(range(len(NAMES)), NAMES)
ax.set_yscale("log")
ax.set_yticks([0.05, 0.1, 0.2, 0.4], ["5%", "10%", "20%", "40%"])
ax.minorticks_off()
ax.set_ylabel("rate on line (deposit)")
ax.set_title("Technical rate on line against the broker's quote")
ax.legend()
fig.tight_layout()
plt.show()

## 13. The decision

The rule uses the return at the quote. The one-price cross-check of section 11 is a second opinion.

* At or above the target (10%): quote the line that we want at the broker's rate.
* Between the floor (6%) and the target: counter at our technical rate on line. If the broker's quote is clearly above the one-price premium (by more than the 5% tolerance), also offer a smaller line at the quote.
* Below the floor: decline on the stand-alone view. But if the quote is clearly above the one-price premium, the shortfall comes from the stand-alone capital standard, not from the price of the risk. Then refer the layer to the portfolio view, and do not decline it.

The rule uses the central estimate. Each verdict prints the 5% to 95% range of the return at the quote across the draws of section 12. It also prints the probability that the quote covers our technical premium. Where that range includes the target or the floor, the verdict is nearer to a coin flip than its words suggest.

In [ ]:
LINE = 0.25  # the share of each layer we would sign if the price is right
SMALL_LINE = 0.10  # the share we would sign at a price that is short of our technical premium but consistent
closing = []
for row in priced:
    above_one_price = row["quoted_total"] >= row["consistent"] * (1 + TOLERANCE)
    if row["quoted_return"] >= RETURN:
        verdict = f"quote {LINE:.0%} at the broker's rate"
    elif row["quoted_return"] >= FLOOR_RETURN:
        verdict = f"counter at {row['rol']:.1%} rate on line"
        if above_one_price:
            verdict += f", or quote {SMALL_LINE:.0%} at the broker's rate"
    elif above_one_price:
        verdict = "refer to the portfolio view (declined on stand-alone capital, priced above one price of risk)"
    else:
        verdict = "decline"
    closing.append(verdict)

print(f"{'layer':>11s} {'burning':>8s} {'experience':>11s} {'exposure':>9s} {'blend':>7s} {'technical':>10s} {'multiple':>9s} "
      f"{'tech. ROL':>10s} {'quoted ROL':>11s} {'return':>7s} {'P(covers)':>10s}")
for row, t, p_cover, b, e, x, bl in zip(priced, terms, cover, burning_cost, experience_loss, exposure_loss, blended):
    print(f"{row['name']:>11s} {money(b):>8s} {money(e):>11s} {money(x):>9s} {money(bl):>7s} {money(row['technical']):>10s} "
          f"{row['technical'] / row['loss']:>9.2f} {row['rol']:>10.1%} {t['quoted_rol']:>11.1%} "
          f"{row['quoted_return']:>7.1%} {p_cover:>10.0%}")
print()
for row, verdict, (low, high), p_cover in zip(priced, closing, return_band, cover):
    print(f"{row['name']:>11s}: {verdict}\n{'':13s}return at the quote {row['quoted_return']:.1%}, {low:.1%} to {high:.1%} "
          f"across the draws; the quote covers our technical premium in {p_cover:.0%} of them")

print()
for row, verdict in zip(priced, closing):
    if verdict.startswith("quote"):
        margin = row["quote"].at(row["quoted_total"]).margin * LINE
        print(f"{row['name']}: a {LINE:.0%} line earns an expected {money(margin)} a year after expenses and brokerage, "
              f"on {money(row['capital'] * LINE)} of capital ({row['quoted_return']:.1%}); "
              f"its 99% TVaR loss is {money(metrics[row['name']]['tvar'] * LINE)}.")

Read the verdicts with their ranges. The working layer is a quote on the central estimate. But its return at the quote is between 7.6% and 15.0% across the draws, and the quote covers our technical premium in 62% of them. This is nearer to a coin flip than the verdict suggests, and supports a moderate line. The range of the middle layer is from just under the floor to about the target. The top layer goes to the portfolio view and is not declined, because its quote is well above the one-price premium. Without the one-price check, the rule declines it.

This is a decision on price. Other items also count. These are the sizes of the lines, the other business of the reinsurer, and the relationship with the cedant. Another item is the accumulation with its other risks: a loss to this program can occur with a loss elsewhere. For the portfolio view, use `Portfolio`, `capital` and the allocation functions of `prospicio.risk` and `prospicio.pricing`, as in `one_year_view.ipynb`.

## 14. How close were the estimates to the truth?

A generator with known parameters made the data (`examples/data/answer_key.json`), so we can check the estimates. In the answer key, the count of claims "over `T`" is for claims on risks insured for more than `T`, as in the listing. The generator mixes the Poisson count of each year with a gamma of coefficient of variation 0.25. Thus the true dispersion of the yearly count is `1 + lambda * 0.25**2`. The layer expected losses are the exact values of the generator under occurrence terms, with no reinstatement or annual limit.

In [ ]:
with open(DATA.parent / "answer_key.json") as f:  # outside the submission's folder
    key = json.load(f)
exposed = sum(w * max(0.0, min(1.0, (hi - T) / (hi - lo)))
              for w, lo, hi in zip(key["limit_band_weights"], lower, upper))  # share of claims on risks insured for more than T
true_lambda = key["lambda_reference"] * (T / key["pareto_reference"]) ** (-key["alpha"]) * exposed
true_dispersion = 1 + true_lambda * key["frequency_mixing_cv"] ** 2
true_layers = [key["layer_expected_loss_2026_occurrence"][n] for n in NAMES]
print(f"{'':32s} {'truth':>10s} {'estimate':>10s} {'error':>8s}")
for label, true_value, estimate in [
        ("Pareto alpha", key["alpha"], ALPHA),
        (f"claims a year over {money(T)}", true_lambda, LAMBDA),
        ("count dispersion (Pearson)", true_dispersion, level_fit.dispersion),
        ("cost trend", key["severity_trend"], SEVERITY_TREND),
        ("exposure growth", key["exposure_growth"], exposure_trend.annual_change),
        ("on-level premium 2025", key["onlevel_premium"]["2025"], onlevel[2025])]:
    fmt = (lambda v: f"{v:,.3f}") if true_value < 100 else (lambda v: money(v))
    print(f"{label:32s} {fmt(true_value):>10s} {fmt(estimate):>10s} {estimate / true_value - 1:>+8.1%}")
print()
print(f"{'expected loss to the layer':32s} " + " ".join(f"{n:>11s}" for n in NAMES))
for label, values in [("truth (occurrence terms)", true_layers), ("burning cost", burning_cost),
                      ("experience model", experience_loss), ("exposure rating", exposure_loss),
                      ("credibility blend", blended)]:
    print(f"{label:32s} " + " ".join(f"{money(v):>11s}" for v in values)
          + ("" if values is true_layers else "   error " + ", ".join(f"{v / t - 1:+.0%}" for v, t in zip(values, true_layers))))

**Reading the comparison.** The premium, exposure and trend inputs are within 0.1% of the truth. Alpha is 7.5% high, inside one standard error (0.17). The frequency is 14% high, which is also about one standard error. A count of 100 claims has a sampling error near 10%, and near 14% when the true dispersion is 2. The frequency also depends on alpha, which sets how far the count of each year moves to `T`. The dispersion is the clear miss: ten yearly counts gave 0.80, and the truth is about 2. For this reason, section 12 tests a clustered count.

At the layers, the experience model is low by 21%, 33% and 39%, more for higher layers. A tail fitted a little too light gives this result. The burning cost has no claim in the top layer. Exposure rating at c=3 is near the truth for the working layer and 13% to 17% low above it, and the blend follows it. All of this is one draw of the generator, and another seed gives other errors. Thus a single run shows the size of the error to expect, not whether a method has a bias. To find a bias, run the calculation over many seeds.

The last step is the most important for pricing. It compares the premium from the true layer losses with the premium from the estimates, with the quotes. The difference is the estimation error from the data and the judgements, as premium. The last lines check the draws of section 12. If the draws describe our uncertainty well, the rate on line from the true losses is usually inside their range.

In [ ]:
# What the same pipeline gives from the true layer losses : the premium we would
# have found with no estimation error in the layer losses.
true_model = fit_references([(limit, att, el) for (limit, att), el in zip(LAYERS, true_layers)], default_alpha=ALPHA)
true_grid, _ = Grid.rounding(true_model.severity, STEP, int(MAX_LOSS / STEP) + 1)
true_annual = program([1.0] * len(terms)).on_grid(claim_count(true_model.frequency, 1.0), true_grid, POINTS)
true_rows = price_layers(true_annual, list(true_annual.expected_reinstatement_premium))
print(f"{'':11s} {'our technical':>14s} {'from true losses':>17s} {'difference':>11s} {'quoted (total)':>14s}")
for row, tr, t in zip(priced, true_rows, terms):
    print(f"{row['name']:>11s} {money(row['technical']):>14s} {money(tr['technical']):>17s} "
          f"{row['technical'] / tr['technical'] - 1:>+11.1%} {money(row['quoted_total']):>14s}")
# Did the draws of section 12 cover the rate on line the true layer losses give?
print()
for tr, (low, high) in zip(true_rows, rol_band):
    inside = "inside" if low <= tr["rol"] <= high else "outside"
    print(f"{tr['name']:>11s}: technical rate on line from the true losses {tr['rol']:.1%}, {inside} the draws' "
          f"5% to 95% range of {low:.1%} to {high:.1%}")

With the true layer losses, the same calculation charges 0.9% less for the working layer and about 8% more for each layer above it. In each layer, the broker's quote is on the same side of both premiums. The quote for the working layer covers both. The quotes for the two layers above are below both. The rates on line from the true losses are inside the 5% to 95% ranges of the draws for all three layers, the top one near the upper end. Thus the draws of section 12 described the uncertainty of this run fairly. Three layers of one run are a weak test of that, for the reason above.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.8))
for (label, values), marker in zip([("burning cost", burning_cost), ("experience model", experience_loss),
                                    ("exposure rating", exposure_loss), ("credibility blend", blended)], "osv^"):
    ax.plot(range(len(NAMES)), [max(v, 1e-3) / 1e6 for v in values], marker, ms=8, label=label)
ax.plot(range(len(NAMES)), [v / 1e6 for v in true_layers], "kD", ms=10, mfc="none", label="truth (the generator)")
ax.set_xticks(range(len(NAMES)), NAMES)
ax.set_yscale("log")
ax.set_ylim(0.1, 3.0)
ax.set_ylabel("expected annual loss to the layer (millions)")
ax.set_title("Expected loss by layer and method (a burning cost of zero is off the chart)")
ax.legend()
fig.tight_layout()
plt.show()

### What to take away

* Check a submission before you model it. The reconciliation is part of the work.
* The large-loss listing needs development, trend and a completeness cut before the fit. The fit must include truncation and censoring: if you ignore them, the tail changes a lot.
* Experience and exposure rating answer the same question from different data. With few claims for the high layers, credibility gives the exposure curve most of the weight. Thus examine the curve and the clustering of claims first. Classical credibility is a conservative weight for a fitted model, so test it.
* A technical premium has a loss, a capital charge, and costs. Compare a broker's quote with it as rate on line, multiple and return at the quote. Show how much the central answer can change.
* A stand-alone capital standard makes high layers expensive. Compare the quote with one price of risk across the program. This separates a layer that does not give a sufficient stand-alone return from a layer that is underpriced as a risk.
* Against the truth, the premium and trend inputs were almost exact. The tail and the frequency were within about one standard error. The layer losses were within 1% to 17%. The estimate missed the dispersion of the counts. One synthetic run shows the size of the errors, not whether a method has a bias.